# GenSafeNav: reproduction of the pretrained-model evaluation

Reproduces the in-distribution test of `Ours_GST` (checkpoint `05207.pt`) from
*Towards Generalizable Safety in Crowd Navigation via Conformal Uncertainty Handling* (CoRL 2025).

**Runtime:** Google Colab, T4 GPU. About 30 minutes in total (the 250-episode test takes about 23 minutes).
See `README.md` for results, issues and limitations.

## 1. Check the GPU and environment

In [ ]:
!nvidia-smi
!python --version

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone the repository

In [ ]:
REPO_URL = "https://github.com/aysannazarmohamady/GenSafeNav-reproduction.git"

!git clone {REPO_URL} /content/GenSafeNav
%cd /content/GenSafeNav

## 3. Build Python-RVO2 (ORCA simulator dependency)

The default build fails under pip build isolation, so it is built without isolation.

In [ ]:
%cd /content/GenSafeNav/Python-RVO2
!pip install -q Cython setuptools wheel cmake
!rm -rf build
!pip install --no-build-isolation . 2>&1 | tail -5
%cd /content/GenSafeNav

import rvo2
print("rvo2 imported:", rvo2)

## 4. Back up the authors' logs

`test.py` overwrites `trained_models/Ours_GST/test/test_05207.pt.log`, so the original is copied first.

In [ ]:
%cd /content/GenSafeNav
!mkdir -p /content/authors_logs
!cp -r trained_models /content/authors_logs/trained_models_original
!grep -E "Testing success rate|Collision cases" /content/authors_logs/trained_models_original/Ours_GST/test/test_05207.pt.log

## 5. Run the pretrained test in the background

`TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1` restores the old `torch.load` behaviour that this checkpoint needs
under PyTorch >= 2.6. Use it only for the authors' own checkpoints.

Note: `test.py` hard-codes `test_size = 250` (one test seed), regardless of `env.test_size` in the config.

In [ ]:
%cd /content/GenSafeNav
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

!nohup python test.py > /content/test_run.log 2>&1 &
print("Started. Run the next cell to check progress.")

## 6. Monitor progress (re-run until it prints `running = 0`)

In [ ]:
!grep -c "ends in" /content/test_run.log | sed 's/^/episodes finished: /'
!tail -n 3 /content/test_run.log
!echo "running = $(ps aux | grep 'test.py' | grep -v grep | wc -l)"

## 7. Summarize the run and compare with the authors' log

In [ ]:
import re

def result_lines(path):
    keep = []
    for line in open(path):
        if "Testing success rate" in line or "Collision cases" in line:
            keep.append(line.strip())
    return keep

mine    = "/content/GenSafeNav/trained_models/Ours_GST/test/test_05207.pt.log"
authors = "/content/authors_logs/trained_models_original/Ours_GST/test/test_05207.pt.log"

print("--- authors' log ---")
print("\n".join(result_lines(authors)))
print("\n--- my run ---")
print("\n".join(result_lines(mine)))


log = open("/content/test_run.log").read()
eps = re.findall(r"Reward=tensor\(\[(-?[\d.]+)\]\)\nEpisode (\d+) ends in (\d+)\n(\w+)", log)
print("\nepisodes parsed:", len(eps))
print("episode | steps | reward")
for r, k, n, outcome in eps:
    if outcome == "Collision":
        print(f"{k:>7} | {n:>5} | {r}")

## 8. Save the results

In [ ]:
!mkdir -p /content/my_results
!cp /content/GenSafeNav/trained_models/Ours_GST/test/test_05207.pt.log /content/my_results/my_run_Ours_GST_250ep.log
!cp /content/test_run.log /content/my_results/stdout_Ours_GST_250ep.log
!ls -la /content/my_results